# Post-Training Results Analysis

This notebook acts as an automated dashboard for analyzing parallel model training jobs. It pulls metrics from Custom UNet JSON logs and standard YOLO CSV logs to plot direct performance comparisons, including Loss progression, Mean Intersection-over-Union (mIoU), and comparative inference hardware footprints.

# Industrial Chignon Detection - Training Analysis Dashboard
### Comparative Study: UNet-ResNet18 vs YOLOv8m-seg vs YOLOv11m-seg

This notebook automatically parses and visualizes the results from the training pipeline executed on Kaggle.

In [ ]:
import os, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

# --- Configuration ---
BASE_PATH = Path("/kaggle/working/outputs")
RESULTS_DIR = BASE_PATH / "results"
LOGS_DIR = RESULTS_DIR / "training_logs"
YOLO_DIR = RESULTS_DIR / "yolo_training"
AUG_DIR = BASE_PATH / "augmented_data"


# --- Output Directory for Analysis Plots (Ensures write-access on Kaggle) ---
SAVE_DIR = Path("/kaggle/working/plots")
SAVE_DIR.mkdir(parents=True, exist_ok=True)


print(f"Analyzing results at: {BASE_PATH}")

## Data Loading & Normalization
Parsing disparate formats (UNet JSON vs YOLO CSV) into a unified DataFrame.

In [ ]:
def load_resnet_epochs(model_name):
    path = LOGS_DIR / model_name
    epochs = []
    for f in sorted(path.glob("epoch_*.json")):
        with open(f, 'r') as j:
            data = json.load(j)
            # Assuming phase 'val' contains the validation metrics for comparison
            if 'val' in data:
                m = data['val']
                epochs.append({
                    'epoch': m['epoch'],
                    'loss': m['loss_mean'],
                    'iou': m['iou_mean'],
                    'dice': m['dice_mean'],
                    'model': model_name
                })
    return pd.DataFrame(epochs)

def load_yolo_epochs(model_name):
    csv_path = YOLO_DIR / model_name / "results.csv"
    if not csv_path.exists(): return pd.DataFrame()
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    return pd.DataFrame({
        'epoch': df['epoch'],
        'loss': df['val/seg_loss'] if 'val/seg_loss' in df else df['val/box_loss'],
        'iou': df['metrics/mAP50-95(M)'] if 'metrics/mAP50-95(M)' in df else df['metrics/mAP50(M)'],
        'dice': df['metrics/mAP50(M)'], # Proxy for high-recall Dice
        'model': model_name
    })

df_unet = load_resnet_epochs("unet_resnet18")
df_v8 = load_yolo_epochs("yolov8m_seg")
df_v11 = load_yolo_epochs("yolov11m_seg")

dfs = [df for df in [df_unet, df_v8, df_v11] if not df.empty]
if dfs:
    full_df = pd.concat(dfs).reset_index(drop=True)
else:
    full_df = pd.DataFrame(columns=['epoch', 'loss', 'iou', 'dice', 'model'])
print(f"Loaded {len(full_df)} epoch records.")

## Performance Comparison
Comparing convergence and accuracy (IoU / Mask Precision) across architectures.

In [ ]:
if full_df.empty:
    print("No training data found to plot!")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    # --- IoU / mAP Over Time ---
    sns.lineplot(data=full_df, x='epoch', y='iou', hue='model', ax=axes[0], marker='o', linewidth=2.5)
    axes[0].set_title("Validation Accuracy (Metric Parallel: IoU / mAP@50-95)")
    axes[0].set_ylabel("Accuracy Score")
    
    # --- Loss Over Time ---
    sns.lineplot(data=full_df, x='epoch', y='loss', hue='model', ax=axes[1], marker='s', linewidth=2.5)
    axes[1].set_title("Validation Loss Curves")
    axes[1].set_yscale('log')
    axes[1].set_ylabel("Loss (Log Scale)")
    
    plt.tight_layout()
    plt.savefig(SAVE_DIR / "comparative_metrics.png", dpi=300)
    plt.show()

## Resource Efficiency & Training Time
Analyzing hardware overhead and deployment feasibility.

In [ ]:
summaries = []
for m in ["unet_resnet18", "yolov8m_seg", "yolov11m_seg"]:
    h_path = LOGS_DIR / m / "training_history.json"
    if h_path.exists():
        with open(h_path, 'r') as j:
            summaries.append(json.load(j))

sdf = pd.DataFrame(summaries)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Training Time ---
sns.barplot(data=sdf, x='model_name', y='total_train_time_sec', ax=axes[0], palette="viridis")
axes[0].set_title("Total Training Duration (Seconds)")
axes[0].set_ylabel("Seconds")

# --- Peak GPU Memory ---
sns.barplot(data=sdf, x='model_name', y='peak_gpu_memory_mb', ax=axes[1], palette="magma")
axes[1].set_title("Peak GPU VRAM Usage (MB)")
axes[1].set_ylabel("Memory (MB)")

plt.tight_layout()
plt.savefig(SAVE_DIR / "hardware_footprint.png", dpi=300)
plt.show()

## Augmented Data Gallery
Visual verification of synthesized training samples.

In [ ]:
aug_samples = sorted(glob.glob(str(AUG_DIR / "*.png")))
if aug_samples:
    indices = np.linspace(0, len(aug_samples)-1, 12, dtype=int)
    plt.figure(figsize=(16, 12))
    for idx, i in enumerate(indices):
        plt.subplot(3, 4, idx+1)
        img = Image.open(aug_samples[i])
        plt.imshow(img)
        title = Path(aug_samples[i]).stem[-30:] # Last 30 chars of name
        plt.title(title, fontsize=8)
        plt.axis('off')
    plt.suptitle("Augmented Data Samples (Transform Preview)", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
else:
    print("No augmented samples found to display.")

## Final Model Recommendation
Summary table formatted for the production deployment decision.

In [ ]:
final_stats = sdf[['model_name', 'best_val_iou', 'total_train_time_sec', 'peak_gpu_memory_mb']]
final_stats.columns = ['Model Architecture', 'Best IoU/mAP', 'Train Time (s)', 'Peak VRAM (MB)']
final_stats = final_stats.sort_values(by='Best IoU/mAP', ascending=False)

def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #90EE90' if v else '' for v in is_max]

final_stats.style.apply(highlight_max, subset=['Best IoU/mAP'])
